# Models - the better way

This version is much like the Models notebook only this one is more to to standard.

In [1]:
'''
This isn't strictly needed. However it solves this annoying pandas error:


/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
  
The problem is solved with xgboost 1.6 but I don't want to use pip in this case and the conda package is currently 1.5.1  
'''

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn import preprocessing
from sklearn import set_config


from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer

from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [3]:
df = pd.read_pickle("../data/diamonds.pkl")

In [4]:
df.dtypes

cut                             object
color                           object
clarity                         object
caret_weight                   float64
cut_quality                     object
lab                             object
symmetry                        object
polish                          object
eye_clean                       object
culet_size                      object
culet_condition                 object
depth_percent                  float64
table_percent                  float64
meas_length                    float64
meas_width                     float64
meas_depth                     float64
girdle_min                      object
girdle_max                      object
fluor_color                     object
fluor_intensity                 object
fancy_color_dominant_color      object
fancy_color_secondary_color     object
fancy_color_overtone            object
fancy_color_intensity           object
total_sales_price                int64
dtype: object

In [5]:
# split into inputs and outputs
X, y = df.drop('total_sales_price', axis=1), df['total_sales_price']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=314)

In [7]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((153792, 24), (153792,), (65911, 24), (65911,))

## Preparing the model

In [8]:
categorical_columns = ['cut', 'color', 'clarity', 'cut_quality', 'symmetry', 'polish',\
                       'lab','eye_clean','culet_size', 'culet_condition', 'lab',\
                       'fancy_color_intensity','fancy_color_dominant_color',\
                       'fancy_color_secondary_color','fancy_color_overtone',\
                       'fluor_color', 'fluor_intensity']
            
for col in categorical_columns:
    print(f" '{col}' has the following values: \n \t {df[col].unique()} \n")

 'cut' has the following values: 
 	 ['Round' 'Pear' 'Oval' 'Marquise' 'Princess' 'Emerald' 'Heart' 'Cushion'
 'Radiant' 'Cushion Modified' 'Asscher'] 

 'color' has the following values: 
 	 ['E' 'F' 'L' 'D' 'J' 'I' 'G' 'H' 'M' 'K' 'unknown'] 

 'clarity' has the following values: 
 	 ['VVS2' 'VVS1' 'I1' 'VS1' 'VS2' 'IF' 'SI2' 'I2' 'SI1' 'SI3' 'I3'] 

 'cut_quality' has the following values: 
 	 ['Excellent' 'Very Good' 'unknown' 'Good' 'Fair' 'Ideal'] 

 'symmetry' has the following values: 
 	 ['Very Good' 'Excellent' 'Good' 'Fair' 'Poor'] 

 'polish' has the following values: 
 	 ['Very Good' 'Excellent' 'Good' 'Fair' 'Poor'] 

 'lab' has the following values: 
 	 ['IGI' 'GIA' 'HRD'] 

 'eye_clean' has the following values: 
 	 ['unknown' 'Yes' 'E1' 'Borderline' 'No'] 

 'culet_size' has the following values: 
 	 ['N' 'unknown' 'S' 'M' 'VS' 'L' 'EL' 'SL' 'VL'] 

 'culet_condition' has the following values: 
 	 ['unknown' 'Abraded' 'Chipped' 'Pointed'] 

 'lab' has the following val

As we noted in the DataMunging notebook, this dataset has numerical, ordinal and categorical columns. So there is a little more work to be done in getting things ready.

#### Numerical 

In [9]:
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
numerical_features 

['caret_weight',
 'depth_percent',
 'table_percent',
 'meas_length',
 'meas_width',
 'meas_depth',
 'total_sales_price']

#### Ordinals

In [10]:
clarity = [['F', 0], ['IF', 1], ['VVS1', 2], ['VVS2', 3], ['VS1', 4], ['VS2', 5],
           ['SI1', 6], ['SI2',7], ['SI3',8], ['I1', 9], ['I2', 10], ['I3', 11]]

In [11]:
cut_quality = [['Ideal', 0],['Excellent', 1], ['Very Good', 2], ['Good', 3],
               ['Fair',4],['None',5],['unknown', 5]]

In [12]:
symmetry = [['Excellent', 0], ['Very Good', 1], ['Good', 2], ['Fair', 3], ['Poor', 4]]

In [13]:
polish = [['Excellent', 0], ['Very Good', 1], ['Good', 2], ['Fair', 3], ['Poor', 4]]

In [14]:
culet_size = [['N', 0], ['VS', 1], ['S', 2], ['M', 3],['SL', 4], ['L', 5], ['VL', 6],
              ['EL', 7], ['unknown', 8]]

In [15]:
ordinal_features = ["clarity", "cut_quality", "symmetry", "polish", 'culet_size']

#### Nominals

In [16]:
all_categoricals = df.select_dtypes(exclude=np.number).columns.to_list()
nominal_features = [cat for cat in all_categoricals if cat not in ordinal_features]

nominal_features

['cut',
 'color',
 'lab',
 'eye_clean',
 'culet_condition',
 'girdle_min',
 'girdle_max',
 'fluor_color',
 'fluor_intensity',
 'fancy_color_dominant_color',
 'fancy_color_secondary_color',
 'fancy_color_overtone',
 'fancy_color_intensity']

## Make and run the pipeline

In [17]:
set_config(display='diagram')

In [18]:
nominal_preprocessor = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore")) ])
 
ordinial_preprocessor = Pipeline(
    steps=[("encoder", OrdinalEncoder()) ])

preprocessor = ColumnTransformer(
    [
        ("nominal", nominal_preprocessor, nominal_features),
        ("ordinal", ordinial_preprocessor, ordinal_features)
    ], remainder='passthrough'
)

In [19]:
pipe = make_pipeline(preprocessor, LinearRegression())
pipe  # click on the diagram below to see the details of each step


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['cut', 'color', 'lab',
                                                   'eye_clean',
                                                   'culet_condition',
                                                   'girdle_min', 'girdle_max',
                                                   'fluor_color',
                                                   'fluor_intensity',
                                                   'fancy_color_dominant_color',
                                                   'fancy_color_secondary_color',
                                                   'fancy_color_overtone',
                                                   'fancy_color_intensity']),
                                                 ('ordinal',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['clarity', 'cut_quality',
                                                   'symmetry', 'polish',
                                                   'culet_size'])])),
                ('linearregression', LinearRegression())])

In [20]:
model = pipe.fit(X_train, y_train)


In [21]:

y_pred = model.predict(X_test)

ValueError: Found unknown categories ['Ideal'] in column 1 during transform

In [ ]:
numeric_features = numerical_features 
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())])

categorical_features = categorical_features
categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder())])

ordinal_features = ordinal_features
ordinal_transformer = Pipeline(
    steps=[("encoder", OrdinalEncoder())])


preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("ord", ordinal_transformer, ordinal_features ),
        ("num", numeric_transformer, numeric_features)
        ])

In [ ]:
print (f'Model r2 score:{r2_score(predictions, y_test)}')

In [ ]:
transformer = QuantileTransformer(output_distribution='normal')
regressor = LinearRegression()

In [ ]:
regr = TransformedTargetRegressor(regressor=regressor, transformer=transformer)

In [ ]:
#regr.fit(X_train, y_train)

In [ ]:
# print("model score: %.3f" % clf.score(X_test, y_test))

In [ ]:
# # 

# tree=DecisionTreeRegressor(max_depth=3)
# tree.fit(X_train, y_train)

# y_pred = tree.predict(X_test)



# print("RMSE: {}".format(np.sqrt(mean_squared_error((y_test),(y_pred)))))
# print("R2  : {}".format(np.sqrt(r2_score((y_test),(y_pred)))))

In [ ]:
# print(f"""
# Decision Trees has a {round((16057.1061-13921.2573)/16057.1061 *100 , 1)} % improvement over baseline in RMSE 
# and a {round((.9262440179416279-.8350244922620658)/.8350244922620658 *100 , 1)} % improvement in R2
# """)

In [ ]:
# xgb_classifier = xgb.XGBClassifier()

In [ ]:
# xgb_r = xgb.XGBRegressor(objective ='reg:squarederror', n_estimators = 1000, seed = 123, verbosity=1)
# xgb_r.fit(X_train, y_train)

# y_pred = xgb_r.predict(X_test)

In [ ]:
# print("RMSE: {}".format(np.sqrt(mean_squared_error((y_test),(y_pred)))))
# print("R2  : {}".format(np.sqrt(r2_score((y_test),(y_pred)))))

### XGBoost regression is strong for two reasons. Good performance and so much faster than Random Forrest Regression. And this is for a basically untuned model.

Grid Search didn't preform as well as expected. In fact it was substantially worse.

Yeah, I don't get that.
```
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000}
Lowest RMSE:  25154.080799724477
```

Let's try a different set of parameters

Started at 5:30pm

```
params = {'learning_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
          'n_estimators': [50, 100, 150, 200, 300],
          'colsample_bytree': [.5,  0.7, 0.8] }

clf = GridSearchCV(estimator=xgb_r, param_grid=params,
                   scoring='neg_mean_squared_error', verbose=1)

clf.fit(X, y)

print("Best parameters:", clf.best_params_)
print("Lowest RMSE: ", (-clf.best_score_)**(1/2.0))
```

Fitting 5 folds for each of 90 candidates, totalling 450 fits <br>
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.6, 'n_estimators': 300} <br>
Lowest RMSE:  25282.458416182304

In [ ]:
# pipe.fit(X_train, y_train)